# Experimento — Análisis Incremental del Pipeline

Cuantifica el aporte individual de cada etapa del pipeline al rendimiento del clasificador LSTM.

| Condición | Descripción |
|:---------:|:------------|
| C0 | Keypoints crudos (sin ninguna etapa) — baseline |
| C2 | C0 + normalización de hombros |
| C3 | C2 + aumento de datos (5 técnicas) — pipeline completo |

> **Nota:** C1 (filtro de calidad) se omite porque `dataset_top35.csv` es un subconjunto de `dataset_limpio.csv`. Todas las muestras ya pasan el filtro — C1 = C0 (confirmado empíricamente: accuracy idéntica en todos los folds).

**Método:** 5-Fold Stratified CV con Wilson CI 95%.  
**Esperado:** C0 < C2 < C3 con la mayor brecha en C2→C3 (aumento).  
**Prerequisito:** `07_extraer_keypoints.py` ejecutado.

In [1]:
import os
os.environ['PYTHONHASHSEED']         = '42'
os.environ['TF_DETERMINISTIC_OPS']   = '1'
os.environ['TF_CUDNN_DETERMINISTIC'] = '1'
os.environ['CUDA_VISIBLE_DEVICES']   = ''   # fuerza CPU para maximo determinismo
os.environ['TF_ENABLE_ONEDNN_OPTS']  = '0'  # evita reduccion multi-hilo no determinista
print('Determinismo activado (CPU forzado, single-thread)')

import sys
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score
from tensorflow.keras.utils import to_categorical
import tensorflow as tf
tf.config.threading.set_intra_op_parallelism_threads(1)
tf.config.threading.set_inter_op_parallelism_threads(1)

sys.path.insert(0, str(Path(os.getcwd()).parent.parent))
from utils.config import (N_FRAMES, N_KEYPOINTS, DATA_DIR,
                           CSV_TOP35, KEYPOINTS_DIR)
from utils.preprocessing import normalizar_hombros
from utils.augmentation import aumentar_secuencia
from utils.model import construir_lstm, callbacks_entrenamiento
from utils.experiment_utils import fijar_semillas, wilson_ci, dividir_train_val as _dividir

SALIDA_DIR = os.path.join(DATA_DIR, "experimentos", "exp_ablacion")
os.makedirs(SALIDA_DIR, exist_ok=True)

N_FOLDS    = 5
MAX_EPOCHS = 150   # protocolo completo (igual que exp1_ab_5fold y exp2_factorial_5fold)
SEED       = 42
VAL_RATIO  = 0.15
BATCH      = 16
HP = dict(lr=0.001, dropout=0.5, regularizacion=0.005, capa_densa=False)

print(f"Directorio de salida: {SALIDA_DIR}")


Determinismo activado (CPU forzado, single-thread)


C:\Users\dell\AppData\Local\Temp\ipykernel_54792\483301000.py:11: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


Directorio de salida: C:\Users\dell\Documents\TIC\Codigo_TIC\experimentos\exp_ablacion


## Utilidades

In [2]:
def dividir_val(X_tr_fold, y_tr_fold, fold):
    """Divide train/val con fallback a ShuffleSplit cuando n_val < n_clases."""
    return _dividir(X_tr_fold, y_tr_fold, seed=SEED + fold, val_ratio=VAL_RATIO)


def entrenar_fold(fold, X_tr, y_tr, X_val, y_val, X_test, y_test, n_clases):
    fijar_semillas(SEED + fold)
    model = construir_lstm(n_clases, **HP)
    hist = model.fit(
        X_tr, to_categorical(y_tr, n_clases),
        validation_data=(X_val, to_categorical(y_val, n_clases)),
        epochs=MAX_EPOCHS, batch_size=BATCH,
        callbacks=callbacks_entrenamiento(patience_stop=20, patience_lr=8),
        verbose=0,
    )
    h       = hist.history
    acc_key = 'accuracy' if 'accuracy' in h else 'categorical_accuracy'
    pred    = np.argmax(model.predict(X_test, verbose=0), axis=1)
    acc     = accuracy_score(y_test, pred)
    ep      = len(h[acc_key])
    tf.keras.backend.clear_session()
    return acc, ep, pred

print("Utilidades definidas.")


Utilidades definidas.


## Carga de datos

In [3]:
def cargar_raw():
    df      = pd.read_csv(CSV_TOP35)
    clases  = sorted(df['glosa'].unique())
    idx_map = {c: i for i, c in enumerate(clases)}
    X, y, video_ids = [], [], []
    omitidos = 0
    for _, row in df.iterrows():
        npy = os.path.join(KEYPOINTS_DIR, f"{row['video_id']}.npy")
        if not os.path.exists(npy): omitidos += 1; continue
        kp = np.load(npy)
        if kp.shape != (N_FRAMES, N_KEYPOINTS): omitidos += 1; continue
        X.append(kp); y.append(idx_map[row['glosa']]); video_ids.append(row['video_id'])
    if omitidos: print(f"  [AVISO] {omitidos} archivos omitidos")
    return (np.array(X, dtype=np.float32), np.array(y, dtype=np.int32), video_ids, clases, idx_map)

X_all, y_all, video_ids, clases, idx_map = cargar_raw()
n_clases = len(clases)
print(f"Dataset top35: {len(X_all)} muestras | {n_clases} clases")

Dataset top35: 153 muestras | 35 clases


## Construcción de condiciones por fold

In [4]:
def construir_train_test(X_all, y_all, idx_trainval, idx_test, condicion, fold):
    """
    C0: keypoints crudos
    C2: + normalizacion de hombros
    C3: + aumento de datos (solo en train)
    """
    X_test_fold = X_all[idx_test]
    y_test_fold = y_all[idx_test]
    X_tv = X_all[idx_trainval]
    y_tv = y_all[idx_trainval]

    if condicion in ('C2', 'C3'):
        X_tv        = np.array([normalizar_hombros(x) for x in X_tv], dtype=np.float32)
        X_test_fold = np.array([normalizar_hombros(x) for x in X_test_fold], dtype=np.float32)

    X_tr, y_tr, X_val, y_val = dividir_val(X_tv, y_tv, fold)

    if condicion == 'C3':
        X_aug, y_aug = [], []
        for xi, yi in zip(X_tr, y_tr):
            for v in aumentar_secuencia(xi):
                X_aug.append(v); y_aug.append(yi)
        X_tr = np.array(X_aug, dtype=np.float32)
        y_tr = np.array(y_aug, dtype=np.int32)

    return X_tr, y_tr, X_val, y_val, X_test_fold, y_test_fold

print("Función de condiciones definida.")

Función de condiciones definida.


## Ejecución 5-Fold por condición

In [ ]:
print("="*65)
print("ANALISIS INCREMENTAL DEL PIPELINE — 5-Fold Stratified CV")
print("="*65)
print("Condiciones: C0 (crudos) | C2 (+norm.) | C3 (+aumento)")

CONDICIONES = ['C0', 'C2', 'C3']
resultados  = {c: [] for c in CONDICIONES}
all_pred    = {c: [] for c in CONDICIONES}
all_true    = []

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

for fold, (idx_trainval, idx_test) in enumerate(skf.split(X_all, y_all)):
    print(f"\n--- FOLD {fold + 1}/{N_FOLDS} ---")
    fold_true = y_all[idx_test].tolist()

    for cond in CONDICIONES:
        X_tr, y_tr, X_val, y_val, X_te, y_te = construir_train_test(
            X_all, y_all,
            idx_trainval.tolist(), idx_test.tolist(),
            cond, fold
        )
        n_orig = len(idx_trainval)
        factor = len(X_tr) // max(n_orig, 1) if cond == 'C3' else 1

        acc, ep, pred = entrenar_fold(fold, X_tr, y_tr, X_val, y_val, X_te, y_te, n_clases)
        n_tag = f"x{factor}" if factor > 1 else f"{len(X_tr)} muestras"
        print(f"  {cond}: acc={acc*100:.1f}%  ep={ep:>3}  train={n_tag}")

        resultados[cond].append({'fold': fold+1, 'condicion': cond,
                                  'accuracy': round(acc,4), 'epocas': ep,
                                  'n_train': len(X_tr), 'n_test': len(X_te)})
        all_pred[cond].extend(pred.tolist())

    all_true.extend(fold_true)

print("\n✓ Análisis incremental completado.")

## Estadísticas finales y aporte de cada etapa

In [ ]:
y_true_arr = np.array(all_true)

DESCRIPCIONES = {
    'C0': 'crudos (baseline)',
    'C2': '+ normalizacion de hombros',
    'C3': '+ aumento de datos (pipeline completo)',
}

print(f"{'='*65}")
print(f"  RESUMEN ANALISIS INCREMENTAL   (N={len(y_true_arr)} muestras evaluadas)")
print(f"{'='*65}")
print(f"  {'Cond':<5} {'Media':>7} {'Std':>6} {'Min':>6} {'Max':>6}  {'IC 95%':>18}  Descripcion")
print(f"  {'-'*75}")

resumen_filas = []
for cond in CONDICIONES:
    accs    = [r['accuracy'] for r in resultados[cond]]
    preds   = np.array(all_pred[cond])
    ci      = wilson_ci(int((preds == y_true_arr).sum()), len(y_true_arr))
    acc_agg = accuracy_score(y_true_arr, preds)
    print(f"  {cond:<5} {np.mean(accs)*100:>6.1f}%"
          f"{np.std(accs)*100:>5.1f}%"
          f"{min(accs)*100:>5.1f}%"
          f"{max(accs)*100:>5.1f}%"
          f"  [{ci[0]*100:.1f}%, {ci[1]*100:.1f}%]"
          f"  {DESCRIPCIONES[cond]}")
    resumen_filas.append({
        'condicion': cond, 'descripcion': DESCRIPCIONES[cond],
        'acc_media': round(np.mean(accs),4), 'acc_std': round(np.std(accs),4),
        'acc_min': round(min(accs),4), 'acc_max': round(max(accs),4),
        'acc_agregada': round(acc_agg,4), 'ci95_low': ci[0], 'ci95_high': ci[1],
        'n_total': len(y_true_arr),
    })

medias = {c: np.mean([r['accuracy'] for r in resultados[c]]) for c in CONDICIONES}
print(f"\n  Aporte de cada etapa del pipeline:")
for i in range(1, len(CONDICIONES)):
    c_prev, c_cur = CONDICIONES[i-1], CONDICIONES[i]
    delta = (medias[c_cur] - medias[c_prev]) * 100
    print(f"    {c_prev} -> {c_cur}: {delta:+.1f} pp  ({DESCRIPCIONES[c_cur]})")
print(f"\n  Mejora total C0 -> C3: {(medias['C3'] - medias['C0'])*100:+.1f} pp")

## Guardar resultados

In [7]:
todos_resultados = []
for cond in CONDICIONES:
    todos_resultados.extend(resultados[cond])

ruta_folds   = os.path.join(SALIDA_DIR, 'resultados_ablacion.csv')
ruta_resumen = os.path.join(SALIDA_DIR, 'comparativa_ablacion.csv')

pd.DataFrame(todos_resultados).to_csv(ruta_folds,   index=False, encoding='utf-8-sig')
df_resumen = pd.DataFrame(resumen_filas)
df_resumen.to_csv(ruta_resumen, index=False, encoding='utf-8-sig')

print(f"Archivos guardados en: {SALIDA_DIR}/")
display(df_resumen)

Archivos guardados en: C:\Users\dell\Documents\TIC\Codigo_TIC\experimentos\exp_ablacion/


,condicion,descripcion,acc_media,acc_std,acc_min,acc_max,acc_agregada,ci95_low,ci95_high,n_total
0,C0,crudos (baseline),0.4043,0.0792,0.3333,0.5161,0.4052,0.3307,0.4844,153
1,C2,+ normalizacion de hombros,0.4181,0.0739,0.3226,0.4839,0.4183,0.3431,0.4975,153
2,C3,+ aumento de datos (pipeline completo),0.5030,0.0817,0.3871,0.6129,0.5033,0.4249,0.5814,153


## Visualización — Efecto acumulativo de cada etapa

In [ ]:
import matplotlib.pyplot as plt

etiquetas  = [f"{c}\n{DESCRIPCIONES[c]}" for c in CONDICIONES]
medias_pct = [medias[c]*100 for c in CONDICIONES]
ci_lows    = [df_resumen[df_resumen['condicion']==c]['ci95_low'].values[0]*100  for c in CONDICIONES]
ci_highs   = [df_resumen[df_resumen['condicion']==c]['ci95_high'].values[0]*100 for c in CONDICIONES]
yerr_low   = [m - l for m, l in zip(medias_pct, ci_lows)]
yerr_high  = [h - m for m, h in zip(medias_pct, ci_highs)]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Barras con IC Wilson 95%
ax = axes[0]
bars = ax.bar(etiquetas, medias_pct,
              color=['#d9534f', '#5bc0de', '#5cb85c'],
              alpha=0.85, yerr=[yerr_low, yerr_high], capsize=5,
              error_kw={'linewidth': 1.5})
ax.set_ylabel('Accuracy media (%)')
ax.set_title('Análisis incremental del pipeline\n(5-Fold CV + Wilson IC 95%)', fontweight='bold')
ax.grid(True, axis='y', alpha=0.3)
for bar, val in zip(bars, medias_pct):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f"{val:.1f}%", ha='center', fontsize=10, fontweight='bold')

# Incrementos
ax2 = axes[1]
incrementos = [0] + [(medias[CONDICIONES[i]] - medias[CONDICIONES[i-1]])*100
                      for i in range(1, len(CONDICIONES))]
etiq_inc = ['C0\nbaseline'] + [f"{CONDICIONES[i-1]}→{CONDICIONES[i]}"
                                for i in range(1, len(CONDICIONES))]
colores2 = ['gray'] + ['forestgreen' if d > 0 else 'tomato' for d in incrementos[1:]]
ax2.bar(etiq_inc, incrementos, color=colores2, alpha=0.85)
ax2.axhline(0, color='black', linewidth=0.8)
ax2.set_ylabel('Incremento (pp)')
ax2.set_title('Aporte de cada etapa del pipeline', fontweight='bold')
ax2.grid(True, axis='y', alpha=0.3)
for i, (eix, v) in enumerate(zip(etiq_inc, incrementos)):
    if v != 0:
        ax2.text(i, v + 0.2, f'{v:+.1f} pp', ha='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(SALIDA_DIR, 'ablacion_pipeline.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Figura guardada.')